<a href="https://colab.research.google.com/github/nestarShaxzod/Revenue-Cost-of-Sales-ETL-Eng/blob/main/Revenue_Cost_of_Sales_ETL_Eng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Revenue and Cost of Sales Reporting Automation**


------------------------------------------------------------------------------
**Objective:** Demonstrate an ETL process for handling 1C accounting data exports using Python (Pandas) and DuckDB SQL to generate an analytical reporting dataset.

Data and source file paths have been anonymized for public release.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import duckdb
pd.set_option('display.float_format', '{:,.2f}'.format)

# ***Input and Output Data Paths***



In [ ]:
path_cost_of_sale = r'"/data/cost_of_sales/"'
path_revenue = r'/data/revenue/'
path_output = r'/output/'

# ***Data Transformation and Processing Functions***

In [ ]:
#______________________________________________Function for converting values to Date format______________________________________________
def def_time_type(df, col):
    df[col] = pd.to_datetime(df[col], format='%d.%m.%Y', dayfirst=True, errors = 'coerce')
    return df

#______________________________________________Function for converting numeric data types________________________________________________
def def_num_type(df, col):
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace(r'\s+', '', regex = True)
    df[col] = df[col].str.replace(',', '.', regex = True)
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    return df

#______________________________Function for merging 1C export files in TXT format_____________________________________
def def_load_and_combine(path_name, search_key = 'Период'):
    file_paths = glob.glob(os.path.join(path_name, '*txt'))
    all_frame = []

    for f_path in file_paths:

      line_skip_row = 0
      with open(f_path, 'r', encoding = 'utf - 8') as f_obj:
        for numb, row_name in enumerate(f_obj):
          if search_key in row_name:
            line_skip_row = numb
            break

      df = pd.read_csv(f_path, encoding = 'utf-8', skiprows=line_skip_row, sep = '\t')
      all_frame.append(df)

    if all_frame:                                                        # Merge all files if any source files were found
          combined_df = pd.concat(all_frame, ignore_index=True)
          return combined_df
    else:
          print(f"Предупреждение: В папке '{path_name}' файлы .txt не найдены.")
          return pd.DataFrame()                                          # Return an empty DataFrame to prevent the script from failing

#__________________________________________ Function for renaming column headers ___________________________________________
def def_col_rename(df):

  columns_name = {'Период': 'Дата',
                'Документ': 'Документ',
                'Аналитика Дт': 'Аналитика_Дт',
                'Аналитика Кт': 'Аналитика_Кт',
                'Дебет': 'Счет_Дт',
                'Кредит': 'Счет_Кт',
                'Unnamed: 6': 'Сумма_Дт',
                'Unnamed: 9': 'Сумма_Кт'}

  df = df.rename(columns=columns_name)
  return df

# ***Cost of Sales: Merging and Transformation of Source Data***

In [ ]:
#______________________________Apply the processing function to Cost of Sales files _____________________________________________________
cdf_cost_of_sale = def_load_and_combine(path_cost_of_sale, search_key = 'Период')
cdf_cost_of_sale = def_col_rename(cdf_cost_of_sale)
cdf_cost_of_sale = def_time_type(cdf_cost_of_sale, 'Дата')
cdf_cost_of_sale = def_num_type(cdf_cost_of_sale, 'Сумма_Дт')
cdf_cost_of_sale = def_num_type(cdf_cost_of_sale, 'Сумма_Кт')

pd.concat([cdf_cost_of_sale.head(5), cdf_cost_of_sale.tail(5)])

# ***Cost of Sales Data Processing using DuckDB SQ***

In [ ]:
#__________________________________________________Data Processing Using DuckDB SQL ____________________________________________________________

# 1. Drop the existing table (if it exists) and recreate it
duckdb.execute("DROP TABLE IF EXISTS cost_of_sale")

create_empty_table = """
CREATE TEMPORARY TABLE cost_of_sale (
    "Дата" DATE,
    "Номер_документа" VARCHAR,
    "Аналитика" VARCHAR,
    "Счет_Дт" VARCHAR,
    "Счет_Кт" VARCHAR,
    "Себес_Сумма" DECIMAL(20,2),
    "Кол-во" DECIMAL(10,5)
);
"""
duckdb.execute(create_empty_table)

insert_query = """
INSERT INTO cost_of_sale
   WITH t1 AS (
    SELECT
        *,
        CASE
            WHEN "Показатель" = 'БУ' AND LEFT("Счет_Дт", 2) = '91' THEN lead("Сумма_Кт") OVER()
            ELSE (lead("Сумма_Кт") OVER())*-1
        END AS "Кол-во"
    FROM cdf_cost_of_sale
    WHERE "Показатель" IN ('БУ', 'Кол.')
)
SELECT
    "Дата",
    STRING_SPLIT("Документ", CHR(10))[1] AS "Номер_документа",
    CASE
       WHEN LEFT("Счет_Дт", 2) = '91' AND "Аналитика_Дт" IS NOT NULL THEN STRING_SPLIT("Аналитика_Дт", CHR(10))[2]
        WHEN LEFT("Счет_Кт", 2) = '91' AND "Аналитика_Кт" IS NOT NULL THEN STRING_SPLIT("Аналитика_Кт", CHR(10))[2]
    END AS "Аналитика",
    "Счет_Дт",
    "Счет_Кт",
    CASE
        WHEN LEFT("Счет_Дт", 2) = '91' THEN "Сумма_Дт"
        WHEN LEFT("Счет_Кт", 2) = '91' THEN "Сумма_Кт" * -1
        ELSE 0
    END AS "Себес_Сумма",
    "Кол-во"
FROM t1
WHERE "Дата" IS NOT NULL AND "Счет_Дт" NOT LIKE '99%' AND "Счет_Кт" NOT LIKE '99%'
ORDER BY "Дата"
"""
duckdb.execute(insert_query)

# 2. Execute the DuckDB query and load the result directly into a new DataFrame
cost_of_sale = duckdb.query("SELECT * FROM cost_of_sale").df()

pd.concat([cost_of_sale.head(5), cost_of_sale.tail(5)])

# ***Revenue: Merging and Transformation of Source Data***

In [ ]:
#_________________________________________Revenue: Data Consolidation and Transformation ___________________________________________
cdf_revenue = def_load_and_combine(path_revenue, search_key = 'Период')
cdf_revenue = def_col_rename(cdf_revenue)
cdf_revenue = def_time_type(cdf_revenue, 'Дата')
cdf_revenue = cdf_revenue.rename(columns={'Unnamed: 5': 'Сумма_Дт', 'Unnamed: 8': 'Сумма_Кт', 'Сумма_Дт': 'Удалить_1', 'Сумма_Кт': 'Удалить_2'})
cdf_revenue = def_num_type(cdf_revenue, 'Сумма_Дт')
cdf_revenue = def_num_type(cdf_revenue, 'Сумма_Кт')

pd.concat([cdf_revenue.head(5), cdf_revenue.tail(5)])

# ***Revenue Data Processing Using DuckDB SQL***

In [ ]:
#__________________________________________________Data Processing in SQL ____________________________________________________________

#Drop the existing table if it already exists, then create a new one
duckdb.execute("DROP TABLE IF EXISTS revenue_SQL")

create_empty_table_revenue = """
    CREATE TEMPORARY TABLE revenue_SQL (
        "Дата" DATE,
        "Номер_документа" VARCHAR,
        "Аналитика" VARCHAR,
        "Счет_Дт" VARCHAR,
        "Счет_Кт" VARCHAR,
        "Выручка_Сумма" DECIMAL(20,2)
        )
"""
duckdb.execute(create_empty_table_revenue)

duckdb.register("duck_cdf_revenue", cdf_revenue)

insert_query_revenue = """
    INSERT INTO revenue_SQL
    SELECT
      "Дата",
       CASE
        WHEN "Счет_Дт" LIKE '90%' THEN STRING_SPLIT("Документ", CHR(10))[1]
        WHEN "Счет_Кт" LIKE '90%' THEN STRING_SPLIT("Документ", CHR(10))[1]
      END AS "Номер_документа",
      CASE
        WHEN "Счет_Дт" LIKE '90%' THEN STRING_SPLIT("Аналитика_Дт", CHR(10))[1]
        WHEN "Счет_Кт" LIKE '90%' THEN STRING_SPLIT("Аналитика_Кт", CHR(10))[1]
      END AS "Аналитика",
      "Счет_Дт",
      "Счет_Кт",
      CASE
        WHEN "Счет_Дт" LIKE '90%' THEN "Сумма_Дт" * -1
        WHEN "Счет_Кт" LIKE '90%' THEN "Сумма_Кт"
      END AS "Выручка_Сумма"
    FROM duck_cdf_revenue
    WHERE ("Счет_Дт" LIKE '90%' OR "Счет_Кт" LIKE '90%') AND ("Счет_Дт" NOT LIKE '99%' AND "Счет_Кт" NOT LIKE '99%')
"""
duckdb.execute(insert_query_revenue)

revenue = duckdb.query("select * from revenue_SQL").df()

pd.concat([revenue.head(5), revenue.tail(5)])

# ***Revenue and Cost of Sales Data Integration (joining) Using DuckDB SQL***

In [ ]:
cost_of_sale['Аналитика'] = cost_of_sale['Аналитика'].str.strip()
revenue['Аналитика'] = revenue['Аналитика'].str.strip()

#__________________________________________________Data Processing in SQL ____________________________________________________________
duckdb.execute("DROP TABLE IF EXISTS group_rev")

group_rev = """
    CREATE TEMPORARY TABLE group_rev AS
    SELECT
      "Дата",
      "Аналитика",
      SUM("Выручка_Сумма") AS "Выручка_Сумма"
    FROM revenue
    GROUP BY
      "Дата",
      "Аналитика"
"""
duckdb.execute(group_rev)

pivot_query = """
    with t1 as (select
      "Дата",
      "Аналитика",
      SUM("Себес_Сумма") AS "Себес_Сумма",
      SUM("Кол-во") AS "Кол-во"
    from cost_of_sale
    group by
      "Дата",
      "Аналитика"),
       t2 as (select
        *
      from t1
      full join group_rev on t1."Дата" = group_rev."Дата" and t1."Аналитика" = group_rev."Аналитика")
        select
          CASE
            when "Дата" is null then "Дата_1"
            else "Дата"
          END AS "Дата",
            CASE
                when "Аналитика" is null then "Аналитика_1"
                else "Аналитика"
            END as "Аналитика",
          "Выручка_Сумма",
          "Себес_Сумма",
          "Кол-во"
        from t2
        order by "Дата", "Выручка_Сумма"
        """
reven_cost_of_sale = duckdb.query(pivot_query).df()

pd.concat([reven_cost_of_sale.head(5), reven_cost_of_sale.tail(5)])

# ***Exporting and Saving the Final Results**

In [ ]:
exl_path = os.path.join(path_output, 'revenue_cost_of_sale.xlsx')

with pd.ExcelWriter(exl_path) as writer:
    reven_cost_of_sale.to_excel(writer, sheet_name='Выр_себес', index=False)
    cost_of_sale.to_excel(writer, sheet_name='Себестоимость', index=False)
    revenue.to_excel(writer, sheet_name='Выручка', index=False)